In [ ]:
import sys
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
import sklearn

print("python", sys.version)
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("opencv-python", cv2.__version__)
print("matplotlib", matplotlib.__version__)
print("scikit-learn", sklearn.__version__)


# Binary Classification Filter
Recommended dataset: **APTOS 2019 Blindness Detection** (medium-sized, image-level labels).


## Repository setup
This notebook expects the repository `src` folder on the Python path.


In [ ]:
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from engine.image_preprocessing import PreprocessConfig, preprocess_fundus_image

data_dir = repo_root / "data"
raw_dir = data_dir / "raw"
processed_dir = data_dir / "processed"
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)


## Kaggle setup (Colab)
Upload your `kaggle.json` file or set `KAGGLE_USERNAME` and `KAGGLE_KEY` before downloading datasets.


In [ ]:
import os
import subprocess
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

if not kaggle_json.exists():
    print("Upload kaggle.json to ~/.kaggle or set KAGGLE_USERNAME/KAGGLE_KEY.")
    print("Colab tip: use files.upload() and then move the file.")

if kaggle_json.exists():
    kaggle_json.chmod(0o600)

# Install the Kaggle CLI if needed
try:
    subprocess.run(["kaggle", "--version"], check=True, capture_output=True)
except (FileNotFoundError, subprocess.CalledProcessError):
    subprocess.run(["pip", "-q", "install", "kaggle==1.6.17"], check=True)


## Download APTOS 2019 from Kaggle
You must accept the competition rules on Kaggle before downloading.


In [ ]:
import zipfile

aptos_competition = "aptos2019-blindness-detection"
aptos_root = raw_dir / "aptos2019"
aptos_root.mkdir(parents=True, exist_ok=True)

if not (aptos_root / "train.csv").exists():
    subprocess.run(["kaggle", "competitions", "download", "-c", aptos_competition, "-p", str(aptos_root)], check=True)
    zip_files = list(aptos_root.glob("*.zip"))
    for zip_file in zip_files:
        with zipfile.ZipFile(zip_file, "r") as zip_ref:
            zip_ref.extractall(aptos_root)
    print("APTOS extracted to", aptos_root)


## Load labels and build splits


In [ ]:
from sklearn.model_selection import train_test_split

train_csv = aptos_root / "train.csv"
train_images = aptos_root / "train_images"

if train_csv.exists():
    labels_df = pd.read_csv(train_csv)
    labels_df["image_path"] = labels_df["id_code"].apply(lambda x: train_images / f"{x}.png")
    labels_df = labels_df[labels_df["image_path"].apply(lambda p: p.exists())]
    labels_df = labels_df.rename(columns={"diagnosis": "label"})
    print(labels_df.head())
else:
    labels_df = pd.DataFrame(columns=["id_code", "label", "image_path"])

if not labels_df.empty:
    train_df, temp_df = train_test_split(
        labels_df, test_size=0.3, random_state=42, stratify=labels_df["label"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"]
    )
    print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))


## Preprocess and visualize a sample


In [ ]:
if not labels_df.empty:
    sample = labels_df.iloc[0]
    image_array = cv2.imread(str(sample["image_path"]))
    if image_array is None:
        print("Failed to read", sample["image_path"])
    else:
        display_config = PreprocessConfig(target_size=(512, 512), normalization="zero_one")
        model_config = PreprocessConfig(target_size=(512, 512), normalization="imagenet")
        display_result = preprocess_fundus_image(sample["image_path"], config=display_config)
        model_result = preprocess_fundus_image(sample["image_path"], config=model_config)
        print("Label:", sample["label"], "Model tensor mean:", float(model_result.image.mean()))
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        original = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)
        axes[0].imshow(original)
        axes[0].set_title("Original")
        axes[0].axis("off")
        axes[1].imshow(display_result.image)
        axes[1].set_title("Preprocessed")
        axes[1].axis("off")
        plt.tight_layout()


## Next step
Train a binary classifier (referable vs non-referable) and log predictions with preprocessing metadata.
